In [1]:
import os
import zipfile

# Directory containing your zip files
source_dir = "/home/sakshi/Unet 3+ reconstruction/Test Data"  # Change to your target directory path if needed

for filename in os.listdir(source_dir):
    if filename.lower().endswith(".zip"):
        zip_path = os.path.join(source_dir, filename)
        
        # Create a folder name corresponding to the zip file (excluding .zip)
        folder_name = os.path.splitext(filename)[0]
        extract_to = os.path.join(source_dir, folder_name)
        
        # Create the destination directory if it doesn't exist
        os.makedirs(extract_to, exist_ok=True)
        
        # Extract the contents
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
            
        print(f"Successfully extracted: {filename} -> {extract_to}/")

Successfully extracted: extracted_images.zip -> /home/sakshi/Unet 3+ reconstruction/Test Data/extracted_images/
Successfully extracted: Malaria_Samples.zip -> /home/sakshi/Unet 3+ reconstruction/Test Data/Malaria_Samples/
Successfully extracted: NILM.zip -> /home/sakshi/Unet 3+ reconstruction/Test Data/NILM/
Successfully extracted: Malaria_Samples_point.zip -> /home/sakshi/Unet 3+ reconstruction/Test Data/Malaria_Samples_point/


In [ ]:
import os
import cv2
import numpy as np
import random
from glob import glob

def pad_to_square(image, target_size, bg_color=255):
    """
    Resizes an image while strictly preserving its aspect ratio.
    Pads the remaining space with the specified background color.
    """
    if isinstance(target_size, tuple):
        target_size = target_size[0]
        
    h, w = image.shape[:2]
    
    # Calculate the scaling factor to fit the target size without stretching
    scale = min(target_size / h, target_size / w)
    new_h, new_w = int(h * scale), int(w * scale)
    
    # Resize the image using the preserved aspect ratio
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
    # Calculate padding for top/bottom and left/right to reach target_size
    top = (target_size - new_h) // 2
    bottom = target_size - new_h - top
    left = (target_size - new_w) // 2
    right = target_size - new_w - left
    
    # Apply the padding
    if len(image.shape) == 3:
        padded = cv2.copyMakeBorder(resized, top, bottom, left, right, 
                                    cv2.BORDER_CONSTANT, value=[bg_color, bg_color, bg_color])
    else:
        padded = cv2.copyMakeBorder(resized, top, bottom, left, right, 
                                    cv2.BORDER_CONSTANT, value=bg_color)
    return padded

def preprocess_anerbc_to_optical_maps(input_dir, output_amp_dir, output_phase_dir, target_size=(256, 256), phi_max=1.0 * np.pi, max_images=None, random_sample=False):
    """
    Processes AneRBC RGB images into amplitude and phase maps using the linear 
    thin-phase object approximation. Saves amplitude and phase into entirely separate folders.
    
    Parameters:
    - input_dir: Path to raw sequential RGB AneRBC image crops.
    - output_amp_dir: Folder path to save the amplitude outputs.
    - output_phase_dir: Folder path to save the phase outputs.
    - target_size: Standardized dimensions for the DLHM simulator (default: 256x256).
    - phi_max: Maximum physical phase delay (e.g., 1.0 * pi for biological cells).
    - max_images: Integer. Maximum number of images to process (e.g., 2000).
    - random_sample: Boolean. If True, picks images randomly. If False, takes the first N alphabetically.
    """
    
    # Ensure the distinct output directories exist
    os.makedirs(output_amp_dir, exist_ok=True)
    os.makedirs(output_phase_dir, exist_ok=True)
    
    # Load all supported image extensions
    image_paths = glob(os.path.join(input_dir, "*.png")) + glob(os.path.join(input_dir, "*.jpg"))
    
    # Apply random shuffle if requested
    if random_sample:
        random.shuffle(image_paths)
    else:
        image_paths = sorted(image_paths)
        
    # Apply the limit if max_images is specified
    if max_images is not None:
        image_paths = image_paths[:max_images]
        
    print(f"Found and selected {len(image_paths)} images from {input_dir}. Starting processing...")
    
    for path in image_paths:
        filename = os.path.basename(path)
        
        # 1. Load image and preserve aspect ratio with padding
        img_bgr = cv2.imread(path)
        if img_bgr is None:
            continue
        img_padded = pad_to_square(img_bgr, target_size=target_size, bg_color=255)
        
        # 2. Target: grayscale the white image to create a single-channel target
        gray = cv2.cvtColor(img_padded, cv2.COLOR_BGR2GRAY)
        
        # 3. Normalize intensity to [0.0, 1.0]
        I = gray.astype(np.float32) / 255.0

        # Apply Background Flattening Threshold
        threshold_value = 0.95 
        I[I > threshold_value] = 1.0
        
        # Apply Edge Margin Masking (Anti-Wrap-Around)
        border_size = 15
        I[:border_size, :] = 1.0  # Top edge
        I[-border_size:, :] = 1.0 # Bottom edge
        I[:, :border_size] = 1.0  # Left edge
        I[:, -border_size:] = 1.0 # Right edge

        # 4. Phase Mapping
        phase = phi_max * (1.0 - I)

        # 5. Amplitude Mapping
        amplitude = np.sqrt(I)
        
        # 6. Save processed maps into their respective separate folders
        base_name = os.path.splitext(filename)[0]
        
        # Save exact high-precision Numpy arrays for the DLHM physics simulator
        np.save(os.path.join(output_amp_dir, f"{base_name}_amp.npy"), amplitude)
        np.save(os.path.join(output_phase_dir, f"{base_name}_phase.npy"), phase)
        
        # Save 8-bit visual maps for quick manual inspection
        amp_visual = (amplitude * 255).astype(np.uint8)
        phase_visual = cv2.normalize(phase, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        
        cv2.imwrite(os.path.join(output_amp_dir, f"{base_name}_amp.png"), amp_visual)
        cv2.imwrite(os.path.join(output_phase_dir, f"{base_name}_phase.png"), phase_visual)

    print(f"Preprocessing completed successfully.")
    print(f" -> Amplitude maps saved to: {output_amp_dir}")
    print(f" -> Phase maps saved to:     {output_phase_dir}")

# ==========================================
# Example usage to execute the function:
# ==========================================
input_folder = r"/home/sakshi/Unet 3+ reconstruction/filtered_1000_dataset/rouleaux"
amplitude_out = r"C:\Unet3+ reconstruction\rauleaux_amplitude"
phase_out = r"C:\Unet3+ reconstruction\rauleaux_phase"

preprocess_anerbc_to_optical_maps(
    input_dir=input_folder, 
    output_amp_dir=amplitude_out, 
    output_phase_dir=phase_out, 
    target_size=(256, 256),              
)

Found and selected 5000 images from C:\Unet3+ reconstruction\Dataset\Thin blood smear images\Rouleaux morphology RBC Sliced. Starting processing...
Preprocessing completed successfully.
 -> Amplitude maps saved to: C:\Unet3+ reconstruction\rauleaux_amplitude
 -> Phase maps saved to:     C:\Unet3+ reconstruction\rauleaux_phase


In [1]:
import os
from pathlib import Path
import random
import shutil

# 1. Define your raw source paths and destination path
SOURCE_DIRS = {
    "anemic_RBC": Path("/home/sakshi/Unet 3+ reconstruction/Dataset/AneRBC dataset/AneRBC_dataset/AneRBC-II/Anemic_individuals/Original_images"),
    "healthy_RBC": Path("/home/sakshi/Unet 3+ reconstruction/Dataset/AneRBC dataset/AneRBC_dataset/AneRBC-II/Healthy_individuals/Original_images"),    
    "rouleaux": Path("/home/sakshi/Unet 3+ reconstruction/Dataset/Rauleaux")
}

OUTPUT_DIR = Path("./filtered_1000_dataset")
TARGET_COUNT = 1000
RANDOM_SEED = 42

# Supported file types in your raw dataset
VALID_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".npz", ".npy"}


def extract_1000_per_class():
  random.seed(RANDOM_SEED)

  for class_name, src_path in SOURCE_DIRS.items():
    if not src_path.exists():
      print(f"[Error] Source directory not found: {src_path}")
      continue

    # Create destination class directory
    dest_dir = OUTPUT_DIR / class_name
    dest_dir.mkdir(parents=True, exist_ok=True)

    # Gather all valid files recursively
    all_files = [
        f
        for f in src_path.rglob("*")
        if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS
    ]

    total_found = len(all_files)
    print(f"[{class_name}] Found {total_found} total files.")

    if total_found == 0:
      print(f"[{class_name}] No valid images found. Skipping.")
      continue

    if total_found < TARGET_COUNT:
      print(
          f"[{class_name}] Warning: Only {total_found} available. Copying"
          " all."
      )
      selected_files = all_files
    else:
      # Uniform random sampling of exactly 1000 files
      selected_files = random.sample(all_files, TARGET_COUNT)

    # Copy files with clean standardized indexing
    for idx, file_path in enumerate(selected_files, start=1):
      dest_file = dest_dir / f"{class_name}_{idx:04d}{file_path.suffix}"
      shutil.copy2(file_path, dest_file)

    print(
        f"[{class_name}] Successfully copied {len(selected_files)} files to"
        f" {dest_dir}\n"
    )

  print("Extraction completed.")


if __name__ == "__main__":
  extract_1000_per_class()

[anemic_RBC] Found 6000 total files.
[anemic_RBC] Successfully copied 1000 files to filtered_1000_dataset/anemic_RBC

[healthy_RBC] Found 6000 total files.
[healthy_RBC] Successfully copied 1000 files to filtered_1000_dataset/healthy_RBC

[rouleaux] Found 12356 total files.
[rouleaux] Successfully copied 1000 files to filtered_1000_dataset/rouleaux

Extraction completed.


In [ ]:
import os
from pathlib import Path
import random
import cv2
from glob import glob
import numpy as np


def pad_to_square(image, target_size=512, bg_color=255):
  """Resizes an image while strictly preserving its aspect ratio.

  Pads the remaining space with the specified background color.
  """
  if isinstance(target_size, tuple):
    target_size = target_size[0]

  h, w = image.shape[:2]

  # Calculate scaling factor
  scale = min(target_size / h, target_size / w)
  new_h, new_w = int(h * scale), int(w * scale)

  # Resize with aspect ratio preserved
  resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)

  # Calculate padding
  top = (target_size - new_h) // 2
  bottom = target_size - new_h - top
  left = (target_size - new_w) // 2
  right = target_size - new_w - left

  # Apply padding
  if len(image.shape) == 3:
    padded = cv2.copyMakeBorder(
        resized,
        top,
        bottom,
        left,
        right,
        cv2.BORDER_CONSTANT,
        value=[bg_color, bg_color, bg_color],
    )
  else:
    padded = cv2.copyMakeBorder(
        resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=bg_color
    )
  return padded


def create_smooth_boundary_window(shape, border_size=15):
  """Generates a 2D cosine-tapered window (Tukey window) to softly blend

  the edges to unity (I=1.0) and prevent high-frequency FFT diffraction
  streaks.
  """
  h, w = shape
  window_h = np.ones(h, dtype=np.float32)
  window_w = np.ones(w, dtype=np.float32)

  # Cosine ramp for edges
  ramp = 0.5 * (1.0 - np.cos(np.linspace(0, np.pi, border_size)))
  window_h[:border_size] = ramp
  window_h[-border_size:] = ramp[::-1]

  window_w[:border_size] = ramp
  window_w[-border_size:] = ramp[::-1]

  window_2d = np.outer(window_h, window_w)
  return window_2d


def preprocess_images_to_optical_maps(
    input_dir,
    output_npz_dir,
    output_preview_dir=None,
    target_size=(512, 512),
    phi_max=1.0 * np.pi,
    max_images=None,
    random_sample=False,
    save_previews=True,
):
  """Processes raw RGB crops into paired amplitude & phase maps.

  - Bundles float32 (amplitude, phase) arrays into a single lossless .npz per
  sample.
  - Applies background flattening and smooth cosine boundary taper.
  - Optionally exports 8-bit PNGs for quick visual inspection.
  """
  input_path = Path(input_dir)
  npz_path = Path(output_npz_dir)
  npz_path.mkdir(parents=True, exist_ok=True)

  if save_previews and output_preview_dir:
    preview_path = Path(output_preview_dir)
    preview_path.mkdir(parents=True, exist_ok=True)
  else:
    preview_path = None

  # Collect all valid images
  valid_extensions = ("*.png", "*.jpg", "*.jpeg")
  image_paths = []
  for ext in valid_extensions:
    image_paths.extend(input_path.glob(ext))

  if random_sample:
    random.shuffle(image_paths)
  else:
    image_paths = sorted(image_paths)

  if max_images is not None:
    image_paths = image_paths[:max_images]

  print(
      f"Found and selected {len(image_paths)} images from {input_path}. Starting"
      " processing..."
  )

  # Precompute the smooth border taper window
  t_size = target_size[0] if isinstance(target_size, tuple) else target_size
  edge_window = create_smooth_boundary_window((t_size, t_size), border_size=15)

  for path in image_paths:
    img_bgr = cv2.imread(str(path))
    if img_bgr is None:
      continue

    # 1. Pad to square preserving aspect ratio
    img_padded = pad_to_square(img_bgr, target_size=t_size, bg_color=255)

    # 2. Convert to grayscale and normalize intensity [0.0, 1.0]
    gray = cv2.cvtColor(img_padded, cv2.COLOR_BGR2GRAY)
    I = gray.astype(np.float32) / 255.0

    # 3. Background flattening
    I[I > 0.95] = 1.0

    # 4. Apply smooth boundary taper to prevent FFT wrap-around artifacts
    # Blend towards background (I = 1.0) at the margins
    I = 1.0 - (1.0 - I) * edge_window

    # 5. Optical field mapping
    phase = (phi_max * (1.0 - I)).astype(np.float32)
    amplitude = np.sqrt(I).astype(np.float32)

    base_name = path.stem

    # 6. Save bundled float32 arrays into a single compressed .npz
    np.savez_compressed(
        npz_path / f"{base_name}.npz", amplitude=amplitude, phase=phase
    )

    # 7. Optional: Save visual PNG previews
    if save_previews and preview_path:
      amp_visual = (amplitude * 255).astype(np.uint8)
      phase_visual = cv2.normalize(
          phase, None, 0, 255, cv2.NORM_MINMAX
      ).astype(np.uint8)

      cv2.imwrite(str(preview_path / f"{base_name}_amp.png"), amp_visual)
      cv2.imwrite(str(preview_path / f"{base_name}_phase.png"), phase_visual)

  print(f"\nProcessing complete:")
  print(f" -> Bundled .npz maps saved to: {npz_path}")
  if save_previews and preview_path:
    print(f" -> PNG inspection previews saved to: {preview_path}")


if __name__ == "__main__":
  input_folder = Path("/home/sakshi/Unet 3+ reconstruction/Malaria_images")
  npz_output = Path("/home/sakshi/Unet 3+ reconstruction/Dataset_processed/Malaria_npz")

  preprocess_images_to_optical_maps(
      input_dir=input_folder,
      output_npz_dir=npz_output,
      target_size=(512, 512),
      phi_max=1.0 * np.pi,
      save_previews=False,  # Saves disk space and finishes 2x faster
  )

Found and selected 779 images from /home/sakshi/Unet 3+ reconstruction/Dataset/Test Data/NILM/NILM. Starting processing...

Processing complete:
 -> Bundled .npz maps saved to: /home/sakshi/Unet 3+ reconstruction/Dataset_processed/Testing_ground_truth/NILM


In [1]:
from glob import glob
import os
from pathlib import Path
import random
import cv2
import numpy as np


def pad_to_square(image, target_size=512, bg_color=255):
  """Resizes an image while strictly preserving its aspect ratio.

  Pads the remaining space with the specified background color.
  """
  if isinstance(target_size, tuple):
    target_size = target_size[0]

  h, w = image.shape[:2]

  # Calculate scaling factor
  scale = min(target_size / h, target_size / w)
  new_h, new_w = int(h * scale), int(w * scale)

  # Resize with aspect ratio preserved
  resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)

  # Calculate padding
  top = (target_size - new_h) // 2
  bottom = target_size - new_h - top
  left = (target_size - new_w) // 2
  right = target_size - new_w - left

  # Apply padding
  if len(image.shape) == 3:
    padded = cv2.copyMakeBorder(
        resized,
        top,
        bottom,
        left,
        right,
        cv2.BORDER_CONSTANT,
        value=[bg_color, bg_color, bg_color],
    )
  else:
    padded = cv2.copyMakeBorder(
        resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=bg_color
    )
  return padded


def create_smooth_boundary_window(shape, border_size=15):
  """Generates a 2D cosine-tapered window (Tukey window) to softly blend

  the edges to unity (I=1.0) and prevent high-frequency FFT diffraction
  streaks.
  """
  h, w = shape
  window_h = np.ones(h, dtype=np.float32)
  window_w = np.ones(w, dtype=np.float32)

  # Cosine ramp for edges
  ramp = 0.5 * (1.0 - np.cos(np.linspace(0, np.pi, border_size)))
  window_h[:border_size] = ramp
  window_h[-border_size:] = ramp[::-1]

  window_w[:border_size] = ramp
  window_w[-border_size:] = ramp[::-1]

  window_2d = np.outer(window_h, window_w)
  return window_2d


def preprocess_images_to_optical_maps(
    input_dir,
    output_npz_dir,
    output_preview_dir=None,
    target_size=(512, 512),
    phi_max=1.0 * np.pi,
    max_images=None,
    random_sample=False,
    save_previews=True,
):
  """Processes raw RGB crops into paired amplitude & phase maps.

  - Bundles float32 (amplitude, phase) arrays into a single lossless .npz per
  sample.
  - Applies background flattening and smooth cosine boundary taper.
  - Optionally exports 8-bit PNGs for quick visual inspection.
  """
  input_path = Path(input_dir)
  npz_path = Path(output_npz_dir)
  npz_path.mkdir(parents=True, exist_ok=True)

  if save_previews and output_preview_dir:
    preview_path = Path(output_preview_dir)
    preview_path.mkdir(parents=True, exist_ok=True)
  else:
    preview_path = None

  # Collect all valid images
  valid_extensions = ("*.png", "*.jpg", "*.jpeg")
  image_paths = []
  for ext in valid_extensions:
    image_paths.extend(input_path.glob(ext))

  if random_sample:
    random.shuffle(image_paths)
  else:
    image_paths = sorted(image_paths)

  if max_images is not None:
    image_paths = image_paths[:max_images]

  print(
      f"Found and selected {len(image_paths)} images from {input_path}. Starting"
      " processing..."
  )

  # Precompute the smooth border taper window
  t_size = target_size[0] if isinstance(target_size, tuple) else target_size
  edge_window = create_smooth_boundary_window((t_size, t_size), border_size=15)

  # Enumerate starting from 1 to create sequential names: malaria_1, malaria_2...
  for idx, path in enumerate(image_paths, start=1):
    img_bgr = cv2.imread(str(path))
    if img_bgr is None:
      continue

    # 1. Pad to square preserving aspect ratio
    img_padded = pad_to_square(img_bgr, target_size=t_size, bg_color=255)

    # 2. Convert to grayscale and normalize intensity [0.0, 1.0]
    gray = cv2.cvtColor(img_padded, cv2.COLOR_BGR2GRAY)
    I = gray.astype(np.float32) / 255.0

    # 3. Background flattening
    I[I > 0.95] = 1.0

    # 4. Apply smooth boundary taper to prevent FFT wrap-around artifacts
    I = 1.0 - (1.0 - I) * edge_window

    # 5. Optical field mapping
    phase = (phi_max * (1.0 - I)).astype(np.float32)
    amplitude = np.sqrt(I).astype(np.float32)

    # Sequential renaming pattern
    base_name = f"malaria_{idx}"

    # 6. Save bundled float32 arrays into a single compressed .npz
    np.savez_compressed(
        npz_path / f"{base_name}.npz", amplitude=amplitude, phase=phase
    )

    # 7. Optional: Save visual PNG previews
    if save_previews and preview_path:
      amp_visual = (amplitude * 255).astype(np.uint8)
      phase_visual = cv2.normalize(
          phase, None, 0, 255, cv2.NORM_MINMAX
      ).astype(np.uint8)

      cv2.imwrite(str(preview_path / f"{base_name}_amp.png"), amp_visual)
      cv2.imwrite(str(preview_path / f"{base_name}_phase.png"), phase_visual)

  print(f"\nProcessing complete:")
  print(f" -> Bundled .npz maps saved to: {npz_path}")
  if save_previews and preview_path:
    print(f" -> PNG inspection previews saved to: {preview_path}")


if __name__ == "__main__":
  input_folder = Path("/home/sakshi/Unet 3+ reconstruction/Malaria_images")
  npz_output = Path(
      "/home/sakshi/Unet 3+ reconstruction/Dataset_processed/Malaria_npz"
  )

  preprocess_images_to_optical_maps(
      input_dir=input_folder,
      output_npz_dir=npz_output,
      target_size=(512, 512),
      phi_max=1.0 * np.pi,
      max_images=1000,  # Limits processing to exactly 1,000 images
      save_previews=False,  # Saves disk space and finishes faster
  )

Found and selected 1000 images from /home/sakshi/Unet 3+ reconstruction/Malaria_images. Starting processing...

Processing complete:
 -> Bundled .npz maps saved to: /home/sakshi/Unet 3+ reconstruction/Dataset_processed/Malaria_npz
